# Reinforcement Learning: Tabular Q-Learning from Scratch
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/10_Reinforcement_Learning/q_learning_intro.ipynb)

RL learns by trial and error: an agent takes actions, receives rewards and updates a value estimate Q(s, a) = expected future reward.

We train an agent to cross FrozenLake (slippery ice!) with pure NumPy + Gymnasium - the classic intro algorithm before DQN/PPO.

In [ ]:
!pip install -q gymnasium

## 1. The environment

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", is_slippery=True, render_mode="ansi")
print("states:", env.observation_space.n, "| actions: 0=left 1=down 2=right 3=up")
state, _ = env.reset(seed=42)
print(env.render())

Reach goal G; holes H end the episode. Slippery means actions sometimes slide sideways - that is why greedy paths fail.

## 2. Q-learning loop

In [ ]:
Q = np.zeros((env.observation_space.n, env.action_space.n))
alpha, gamma, eps = 0.9, 0.95, 1.0
rewards_history = []

for ep in range(5000):
    state, _ = env.reset()
    total, done = 0.0, False
    while not done:
        if np.random.rand() < eps:                 # explore
            action = env.action_space.sample()
        else:                                      # exploit
            action = int(np.argmax(Q[state]))
        nxt, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        # Bellman update - THE line of Q-learning:
        Q[state, action] += alpha * (reward + gamma * np.max(Q[nxt]) - Q[state, action])
        state, total = nxt, total + reward
    eps = max(0.01, eps * 0.999)                   # decay exploration
    rewards_history.append(total)

import matplotlib.pyplot as plt
window = 100
smooth = np.convolve(rewards_history, np.ones(window) / window, mode="valid")
plt.plot(smooth); plt.xlabel("episode"); plt.ylabel(f"avg reward ({window}-ep)")
plt.title("Learning curve"); plt.show()

## 3. Inspect what the agent learned

In [ ]:
import seaborn as sns
plt.figure(figsize=(6, 5))
sns.heatmap(Q, annot=True, fmt=".2f", cmap="YlGnBu",
            xticklabels=["L", "D", "R", "U"])
plt.title("Q-table (value of each state-action)"); plt.show()

print("greedy path:")
s, _ = env.reset(); done = False
while not done:
    s, r, term, trunc, _ = env.step(int(np.argmax(Q[s])))
    done = term or trunc
    print(env.render())

## 4. Evaluate greedy policy over 200 episodes

In [ ]:
wins = 0
for _ in range(200):
    s, _ = env.reset(); done = False
    while not done:
        s, r, term, trunc, _ = env.step(int(np.argmax(Q[s])))
        done = term or trunc
    wins += r
print(f"success rate: {wins/200:.0%}")

## Where to go next
| Concept | Algorithm |
|---|---|
| too many states for a table | DQN (deep Q-network) |
| continuous actions | PPO / SAC (stable-baselines3) |
| game self-play | AlphaZero-style MCTS |

Same loop though: act -> observe reward -> update value -> repeat.